# EDA and Baseline Models

This notebook loads the breast cancer survival dataset, explores class imbalance,
and establishes baseline Random Forest models before any sampling or advanced
ensemble techniques. **False negatives (missed Dead patients) are the primary
clinical concern** throughout this project.

## 1. Setup and data loading

In [1]:
import os
import sys
import time

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

sys.path.insert(0, os.path.abspath('..'))

assert os.path.exists(os.path.join('..', 'data', 'Breast_Cancer.csv')), \
    "Dataset not found. Place Breast_Cancer.csv in the data/ folder."

from src.preprocessing import load_and_clean, split_features_target, get_train_test_split
from src.feature_engineering import add_features
from src.models import build_model, train_model
from src.evaluation import get_all_metrics, save_results, evaluate_model
from src import visualization

DATA_PATH = os.path.join('..', 'data', 'Breast_Cancer.csv')
FIGURES_DIR = os.path.join('..', 'results', 'figures')
METRICS_DIR = os.path.join('..', 'results', 'metrics')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

df = load_and_clean(DATA_PATH)
df = add_features(df)
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = get_train_test_split(X, y)

print(f"Dataset shape after encoding: {X.shape}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

Dataset shape after encoding: (4024, 30)
Train: 3219 | Test: 805


## 2. Class distribution

The dataset is **imbalanced**: Alive patients dominate. A naive classifier can
achieve high accuracy while missing most Dead cases.

In [2]:
counts = y.value_counts().sort_index()
print(counts)
print()
print((counts / len(y) * 100).round(2).astype(str) + '%')

visualization.plot_class_distribution(
    y,
    save_path=os.path.join(FIGURES_DIR, '01_class_distribution.png')
)
print('Saved: results/figures/01_class_distribution.png')

status
0     616
1    3408
Name: count, dtype: int64

status
0    15.31%
1    84.69%
Name: count, dtype: str


Saved: results/figures/01_class_distribution.png


## 3. Dataset overview

The 16 original columns describe demographics, tumour staging, receptor status,
and lymph node involvement. **`survival_months` is dropped** because it is
directly related to the target (Alive/Dead) and would cause **data leakage** if
used as a feature.

In [3]:
raw_df = pd.read_csv(DATA_PATH)
raw_df.columns = raw_df.columns.str.strip().str.lower().str.replace(' ', '_', regex=False)

print('Column names and roles:')
roles = {
    'age': 'Patient age',
    'race': 'Race/ethnicity',
    'marital_status': 'Marital status',
    't_stage': 'Tumour stage (T)',
    'n_stage': 'Node stage (N)',
    '6th_stage': 'Combined 6th edition stage',
    'differentiate': 'Differentiation grade',
    'grade': 'Histologic grade',
    'a_stage': 'Anatomic stage',
    'tumor_size': 'Tumour size (mm)',
    'estrogen_status': 'ER receptor status',
    'progesterone_status': 'PR receptor status',
    'regional_node_examined': 'Lymph nodes examined',
    'reginol_node_positive': 'Positive lymph nodes',
    'survival_months': 'Months survived (DROPPED — leakage)',
    'status': 'Target: Alive / Dead',
}
for col, role in roles.items():
    print(f'  {col:30s} → {role}')

print()
raw_df.info()
print()
numeric_cols = raw_df.select_dtypes(include='number').columns
raw_df[numeric_cols].describe().T

Column names and roles:
  age                            → Patient age
  race                           → Race/ethnicity
  marital_status                 → Marital status
  t_stage                        → Tumour stage (T)
  n_stage                        → Node stage (N)
  6th_stage                      → Combined 6th edition stage
  differentiate                  → Differentiation grade
  grade                          → Histologic grade
  a_stage                        → Anatomic stage
  tumor_size                     → Tumour size (mm)
  estrogen_status                → ER receptor status
  progesterone_status            → PR receptor status
  regional_node_examined         → Lymph nodes examined
  reginol_node_positive          → Positive lymph nodes
  survival_months                → Months survived (DROPPED — leakage)
  status                         → Target: Alive / Dead

<class 'pandas.DataFrame'>
RangeIndex: 4024 entries, 0 to 4023
Data columns (total 16 columns):
 #   Colum

,count,mean,std,min,25%,50%,75%,max
age,4024.0,53.972167,8.963134,30.0,47.0,54.0,61.0,69.0
tumor_size,4024.0,30.473658,21.119696,1.0,16.0,25.0,38.0,140.0
regional_node_examined,4024.0,14.357107,8.099675,1.0,9.0,14.0,19.0,61.0
reginol_node_positive,4024.0,4.158052,5.109331,1.0,1.0,2.0,5.0,46.0
survival_months,4024.0,71.297962,22.921430,1.0,56.0,73.0,90.0,107.0


## 4. Vanilla RF — no balancing (true baseline)

A standard Random Forest with no `class_weight` and no resampling. This shows
how the model behaves when we ignore imbalance entirely.

In [4]:
print('=' * 60)
print('TRAINING: Vanilla Random Forest (no balancing)')
print('=' * 60)

vanilla_metrics = None
vanilla_model = None

try:
    vanilla_model = build_model('vanilla_rf')
    start = time.time()
    vanilla_model = train_model(vanilla_model, X_train, y_train)
    print(f'Trained in {time.time() - start:.1f}s')

    y_prob_vanilla = vanilla_model.predict_proba(X_test)[:, 1]
    y_pred_vanilla = vanilla_model.predict(X_test)
    vanilla_metrics = get_all_metrics(
        'Vanilla RF', y_test, y_pred_vanilla, y_prob_vanilla, threshold=0.5
    )
    print(pd.Series(vanilla_metrics))
    print('\nConfusion matrix:')
    print(confusion_matrix(y_test, y_pred_vanilla, labels=[0, 1]))
    print('\nClassification report:')
    print(classification_report(y_test, y_pred_vanilla, target_names=['Dead', 'Alive']))
except Exception as e:
    print(f'ERROR training Vanilla RF: {e}')

TRAINING: Vanilla Random Forest (no balancing)


Trained in 3.9s


model_name                         Vanilla RF
threshold                                 0.5
accuracy                               0.8373
recall_dead                            0.1138
precision_dead                         0.3889
f1_dead                                0.1761
f2_dead                                0.1326
roc_auc                                0.7018
pr_auc                                 0.9223
false_negatives                           109
recall_alive                           0.9677
precision_alive                        0.8583
f1_alive                               0.9097
timestamp          2026-06-16T12:25:00.490618
dtype: object

Confusion matrix:
[[ 14 109]
 [ 22 660]]

Classification report:
              precision    recall  f1-score   support

        Dead       0.39      0.11      0.18       123
       Alive       0.86      0.97      0.91       682

    accuracy                           0.84       805
   macro avg       0.62      0.54      0.54       805
weig

## 5. Baseline RF — with `class_weight='balanced'`

The same hyperparameters, but sklearn re-weights classes inversely to their
frequency. This is a simple algorithm-level fix with no data resampling.

In [5]:
print('=' * 60)
print("TRAINING: Random Forest (class_weight='balanced')")
print('=' * 60)

balanced_metrics = None

try:
    balanced_model = build_model('vanilla_rf', class_weight='balanced')
    start = time.time()
    balanced_model = train_model(balanced_model, X_train, y_train)
    print(f'Trained in {time.time() - start:.1f}s')

    balanced_metrics = evaluate_model(
        'RF class_weight=balanced', balanced_model, X_test, y_test, threshold=0.5
    )
    print(pd.Series(balanced_metrics))

    if vanilla_metrics:
        compare = pd.DataFrame([vanilla_metrics, balanced_metrics]).set_index('model_name')
        print('\nComparison (key metrics):')
        print(compare[['recall_dead', 'precision_dead', 'f1_dead', 'false_negatives', 'pr_auc']])
except Exception as e:
    print(f'ERROR training Balanced RF: {e}')

TRAINING: Random Forest (class_weight='balanced')


Trained in 4.0s


model_name           RF class_weight=balanced
threshold                                 0.5
accuracy                                0.846
recall_dead                            0.0894
precision_dead                         0.4783
f1_dead                                0.1507
f2_dead                                0.1068
roc_auc                                0.7034
pr_auc                                 0.9246
false_negatives                           112
recall_alive                           0.9824
precision_alive                        0.8568
f1_alive                               0.9153
timestamp          2026-06-16T12:25:05.611918
dtype: object

Comparison (key metrics):
                          recall_dead  precision_dead  f1_dead  \
model_name                                                       
Vanilla RF                     0.1138          0.3889   0.1761   
RF class_weight=balanced       0.0894          0.4783   0.1507   

                          false_negatives  pr_auc 

## 6. SVM reference result (from original notebook)

These metrics come from a previously tuned SVM on the same dataset. They serve
as an external reference point — not re-trained here.

In [6]:
from datetime import datetime

svm_metrics = {
    'model_name': 'SVM (reference)',
    'threshold': 0.5,
    'accuracy': 0.7035,
    'roc_auc': 0.729,
    'pr_auc': 0.361,
    'recall_dead': 0.585,
    'precision_dead': 0.282,
    'f1_dead': 0.380,
    'f2_dead': round((5 * 0.282 * 0.585) / (4 * 0.282 + 0.585), 4),
    'false_negatives': 51,
    'recall_alive': None,
    'precision_alive': None,
    'f1_alive': None,
    'timestamp': datetime.now().isoformat(),
}
print('SVM reference metrics (from original tuned notebook):')
print(pd.Series(svm_metrics))

SVM reference metrics (from original tuned notebook):
model_name                    SVM (reference)
threshold                                 0.5
accuracy                               0.7035
roc_auc                                 0.729
pr_auc                                  0.361
recall_dead                             0.585
precision_dead                          0.282
f1_dead                                  0.38
f2_dead                                0.4815
false_negatives                            51
recall_alive                             None
precision_alive                          None
f1_alive                                 None
timestamp          2026-06-16T12:25:05.649295
dtype: object


## 7. Save baseline results

In [7]:
baseline_results = []
if vanilla_metrics:
    baseline_results.append(vanilla_metrics)
if balanced_metrics:
    baseline_results.append(balanced_metrics)
baseline_results.append(svm_metrics)

save_path = os.path.join(METRICS_DIR, 'baseline_results.csv')
save_results(baseline_results, save_path)
pd.DataFrame(baseline_results)

Results saved to ..\results\metrics\baseline_results.csv (3 rows)


,model_name,threshold,accuracy,recall_dead,precision_dead,f1_dead,f2_dead,roc_auc,pr_auc,false_negatives,recall_alive,precision_alive,f1_alive,timestamp
0,Vanilla RF,0.5,0.8373,0.1138,0.3889,0.1761,0.1326,0.7018,0.9223,109,0.9677,0.8583,0.9097,2026-06-16T12:25:00.490618
1,RF class_weight=balanced,0.5,0.8460,0.0894,0.4783,0.1507,0.1068,0.7034,0.9246,112,0.9824,0.8568,0.9153,2026-06-16T12:25:05.611918
2,SVM (reference),0.5,0.7035,0.5850,0.2820,0.3800,0.4815,0.7290,0.3610,51,NaN,NaN,NaN,2026-06-16T12:25:05.649295
